# 02 — Marketing Metrics Engine
**Purpose:** Calculate aggregate performance metrics by Channel, Campaign Type, and Month, and generate SQL equivalence queries.
**Inputs:** `data/processed/clean.csv` (200,000 rows × 29 columns)
**Outputs:** `data/processed/summary_channel.csv`, `data/processed/summary_campaign.csv`, `data/processed/summary_month.csv`, `sql/metrics.sql`

> ⚠️ All ratios are computed from summed totals. All revenue figures are SIMULATED.


## 1. Imports & Setup


In [1]:
# Path Resolver Helper: Ensures notebook runs seamlessly from both root and notebooks/ directories
import os
from pathlib import Path

BASE_DIR = Path('..') if Path('../data').exists() else Path('.')
DATA_RAW = BASE_DIR / 'data' / 'raw'
DATA_PROC = BASE_DIR / 'data' / 'processed'
DATA_SMPL = BASE_DIR / 'data' / 'sample'
IMAGE_DIR = BASE_DIR / 'images'
SQL_DIR = BASE_DIR / 'sql'
DASH_DIR = BASE_DIR / 'dashboard' / 'data'

for p in [DATA_RAW, DATA_PROC, DATA_SMPL, IMAGE_DIR, SQL_DIR, DASH_DIR]:
    p.mkdir(parents=True, exist_ok=True)

import pandas as pd
import numpy as np
from pathlib import Path

np.random.seed(42)
SEED = 42

PRO_DIR = Path('../data/processed')
SQL_DIR = Path('../sql')
SQL_DIR.mkdir(exist_ok=True)

print(f"pandas {pd.__version__} | numpy {np.__version__}")


pandas 3.0.5 | numpy 2.5.2


## 2. Load Clean Data & Verify Totals

We immediately record the grand totals from `clean.csv`.  
Every summary table produced in this notebook must reconcile to these numbers — verified in Section 7.


In [2]:
df = pd.read_csv(PRO_DIR / 'clean.csv')
df['date'] = pd.to_datetime(df['date'])

print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Date range: {df['date'].min().date()} to {df['date'].max().date()}")
print()

# ── Grand totals (the sanity-check anchors) ──────────────────────────────
GRAND = {
    'impressions'  : df['impressions'].sum(),
    'clicks'       : df['clicks'].sum(),
    'conversions'  : df['conversions'].sum(),
    'spend'        : df['spend'].sum(),
    'revenue'      : df['revenue'].sum(),
    'n_campaigns'  : len(df),
}
print("=== Grand Totals (Sanity Anchors) ===")
for k, v in GRAND.items():
    print(f"  {k:15s}: {v:>22,.2f}")


Shape: 200,000 rows x 29 columns
Date range: 2021-01-01 to 2021-12-31

=== Grand Totals (Sanity Anchors) ===
  impressions    :       1,101,460,304.00
  clicks         :         109,954,406.00
  conversions    :           8,804,727.00
  spend          :       2,500,878,608.00
  revenue        :       2,654,825,117.12
  n_campaigns    :             200,000.00


## 3. Reusable Metric Functions

Each function takes **pre-aggregated sums** (not row-level values) and returns a scalar.  
Every division is guarded against zero with `np.where` or a conditional.

### Formula Reference

| Metric | Formula | What It Measures |
|--------|---------|-----------------|
| CTR | Clicks / Impressions | Share of ad viewers who clicked |
| CVR | Conversions / Clicks | Share of clickers who converted |
| CPC | Spend / Clicks | Average cost per click |
| CPA | Spend / Conversions | Average cost to acquire one customer |
| CAC | Spend / New Customers | Acquisition cost for **new** customers only |
| ROAS | Revenue / Spend | Revenue generated per $1 of spend |
| ROI | (Revenue − Spend) / Spend | Profit returned per $1 invested |
| Gross Profit | Revenue × Margin | Actual profit after COGS |
| LTV | Effective AOV × Avg Orders × Margin | Lifetime value per customer |
| LTV:CAC | LTV / CAC | How many $ returned per $ acquisition cost |
| Payback | CAC / (Monthly Revenue per Customer × Margin) | Months to recoup acquisition cost |

> **CAC assumption:** No `new_customer` column exists in the dataset.  
> We assume **new customers = conversions × 0.60** (i.e., 60% of converters are new customers).  
> This is a documented assumption — real businesses track new vs. returning separately.  
> To change it, update `NEW_CUSTOMER_SHARE` below.


In [3]:
# ── Global assumption constants ─────────────────────────────────────────
NEW_CUSTOMER_SHARE  = 0.60   # 60% of conversions assumed to be new customers
GROSS_MARGIN        = 0.40   # 40% gross margin (industry avg for e-commerce)
AVG_ORDERS_PER_CUST = 2.50   # Average lifetime orders per customer (assumption)
AVG_MONTHS_ACTIVE   = 12.0   # Customer active window for payback calculation

# ─────────────────────────────────────────────────────────────────────────
# Core metric functions — all accept aggregated (summed) inputs
# ─────────────────────────────────────────────────────────────────────────

def calc_ctr(clicks, impressions):
    """CTR = Clicks / Impressions
    Fraction of ad viewers who clicked. Higher = more compelling creative."""
    return clicks / impressions if impressions > 0 else np.nan

def calc_cvr(conversions, clicks):
    """CVR = Conversions / Clicks
    Fraction of clickers who completed the desired action. Higher = better landing page / offer."""
    return conversions / clicks if clicks > 0 else np.nan

def calc_cpc(spend, clicks):
    """CPC = Spend / Clicks
    Average cost paid per click. Lower = more efficient traffic."""
    return spend / clicks if clicks > 0 else np.nan

def calc_cpa(spend, conversions):
    """CPA = Spend / Conversions
    Average total spend required to produce one conversion.
    Derived from budget, not CPA bid — see README for distinction."""
    return spend / conversions if conversions > 0 else np.nan

def calc_cac(spend, conversions, new_customer_share=NEW_CUSTOMER_SHARE):
    """CAC = Spend / New Customers
    New customers estimated as: conversions * new_customer_share (default 60%).
    ASSUMPTION: 60% of conversions are first-time customers.
    Source: industry benchmark for digital marketing (HubSpot, 2022)."""
    new_customers = conversions * new_customer_share
    return spend / new_customers if new_customers > 0 else np.nan

def calc_roas(revenue, spend):
    """ROAS = Revenue / Spend
    Revenue generated per $1 of spend. ROAS = 1.0 means breakeven.
    Industry benchmark: ROAS > 4x is considered strong for paid social."""
    return revenue / spend if spend > 0 else np.nan

def calc_roi(revenue, spend):
    """ROI = (Revenue - Spend) / Spend
    Profit returned per $1 invested.
    ROI = 0 means breakeven; ROI = -1.0 means 100% loss."""
    return (revenue - spend) / spend if spend > 0 else np.nan

def calc_gross_profit(revenue, margin=GROSS_MARGIN):
    """Gross Profit = Revenue * Margin
    Profit after cost of goods. Does NOT include marketing spend.
    ASSUMPTION: 40% gross margin across all channels and segments."""
    return revenue * margin

def calc_effective_aov(revenue, conversions):
    """Effective AOV = Revenue / Conversions
    Average order value derived from aggregated sums (not the simulated row-level AOV).
    More reliable than averaging row-level AOV, especially across channel cuts."""
    return revenue / conversions if conversions > 0 else np.nan

def calc_ltv(revenue, conversions,
             avg_orders=AVG_ORDERS_PER_CUST, margin=GROSS_MARGIN):
    """LTV = Effective_AOV * Avg_Orders_Per_Customer * Gross_Margin
    Estimated revenue a single customer generates over their lifetime.
    ASSUMPTIONS:
      - avg_orders = 2.5 repeat purchases per customer
      - margin = 40% gross margin
    These are illustrative. Real LTV requires cohort retention data."""
    effective_aov = calc_effective_aov(revenue, conversions)
    if effective_aov is None or np.isnan(effective_aov):
        return np.nan
    return effective_aov * avg_orders * margin

def calc_ltv_cac_ratio(ltv, cac):
    """LTV:CAC = LTV / CAC
    How many dollars of lifetime value are generated per dollar of acquisition cost.
    Benchmark: LTV:CAC >= 3.0 is considered healthy (David Skok, SaaS benchmarks)."""
    return ltv / cac if (cac and cac > 0) else np.nan

def calc_payback_months(cac, revenue, conversions,
                        avg_months=AVG_MONTHS_ACTIVE, margin=GROSS_MARGIN):
    """Payback Period = CAC / (Monthly Revenue per Customer * Margin)
    Number of months to recoup the cost of acquiring one customer.
    ASSUMPTIONS:
      - Monthly revenue per customer = Effective_AOV / avg_months_active
      - avg_months_active = 12 months
      - Gross margin = 40%
    Shorter payback = healthier business."""
    effective_aov = calc_effective_aov(revenue, conversions)
    if not cac or cac <= 0 or effective_aov is None or np.isnan(effective_aov):
        return np.nan
    monthly_rev_per_cust = effective_aov / avg_months
    monthly_profit_per_cust = monthly_rev_per_cust * margin
    return cac / monthly_profit_per_cust if monthly_profit_per_cust > 0 else np.nan

# Quick smoke test
print("Smoke test (100 clicks, 10 conversions, $1000 spend, $2500 revenue):")
print(f"  CTR  = {calc_ctr(100, 1000):.4f}")
print(f"  CVR  = {calc_cvr(10, 100):.4f}")
print(f"  CPC  = ${calc_cpc(1000, 100):.2f}")
print(f"  CPA  = ${calc_cpa(1000, 10):.2f}")
print(f"  CAC  = ${calc_cac(1000, 10):.2f}  (60% new customer share)")
print(f"  ROAS = {calc_roas(2500, 1000):.2f}x")
print(f"  ROI  = {calc_roi(2500, 1000):.4f}  ({calc_roi(2500, 1000)*100:.1f}%)")
ltv = calc_ltv(2500, 10)
cac = calc_cac(1000, 10)
print(f"  LTV  = ${ltv:.2f}")
print(f"  LTV:CAC = {calc_ltv_cac_ratio(ltv, cac):.2f}x")
print(f"  Payback = {calc_payback_months(cac, 2500, 10):.1f} months")


Smoke test (100 clicks, 10 conversions, $1000 spend, $2500 revenue):
  CTR  = 0.1000
  CVR  = 0.1000
  CPC  = $10.00
  CPA  = $100.00
  CAC  = $166.67  (60% new customer share)
  ROAS = 2.50x
  ROI  = 1.5000  (150.0%)
  LTV  = $250.00
  LTV:CAC = 1.50x
  Payback = 20.0 months


## 4. The `summarize()` Function

**Design:** Sum raw counts first → then compute all ratios from sums. Never average ratios.

```
summarize(df, group_cols)
   |
   +-- Step 1: GROUP BY group_cols, SUM all raw additive columns
   |           (impressions, clicks, conversions, spend, revenue, n_campaigns)
   |
   +-- Step 2: COMPUTE all derived metrics from the sums
               CTR, CVR, CPC, CPA, CAC, ROAS, ROI,
               gross_profit, effective_aov, ltv, ltv_cac, payback_months
```

This function is the single source of truth for all three summary tables.


In [4]:
def summarize(df, group_cols):
    """
    Aggregate marketing metrics for any group dimension.

    Parameters
    ----------
    df         : pd.DataFrame — clean.csv with raw additive columns
    group_cols : list[str]   — columns to group by (e.g. ['channel'])

    Returns
    -------
    pd.DataFrame with one row per group, containing:
      Raw sums: n_campaigns, impressions, clicks, conversions, spend, revenue
      Derived ratios (computed from sums, NOT averaged): 
        ctr, cvr, cpc, cpa, cac, roas, roi_pct,
        gross_profit, effective_aov, ltv, ltv_cac_ratio, payback_months
    """
    # ── Step 1: Sum all additive raw columns ─────────────────────────────
    agg = df.groupby(group_cols, observed=True).agg(
        n_campaigns   = ('campaign_id',  'count'),
        impressions   = ('impressions',  'sum'),
        clicks        = ('clicks',       'sum'),
        conversions   = ('conversions',  'sum'),
        spend         = ('spend',        'sum'),
        revenue       = ('revenue',      'sum'),
        engagement_score_mean = ('engagement_score', 'mean'),  # engagement: mean is acceptable
    ).reset_index()

    # ── Step 2: Derive all ratios from sums ──────────────────────────────
    # Guard: every function already handles zero denominators internally
    agg['ctr']            = agg.apply(lambda r: calc_ctr(r['clicks'],      r['impressions']),  axis=1)
    agg['cvr']            = agg.apply(lambda r: calc_cvr(r['conversions'], r['clicks']),        axis=1)
    agg['cpc']            = agg.apply(lambda r: calc_cpc(r['spend'],       r['clicks']),        axis=1)
    agg['cpa']            = agg.apply(lambda r: calc_cpa(r['spend'],       r['conversions']),   axis=1)
    agg['cac']            = agg.apply(lambda r: calc_cac(r['spend'],       r['conversions']),   axis=1)
    agg['roas']           = agg.apply(lambda r: calc_roas(r['revenue'],    r['spend']),          axis=1)
    agg['roi']            = agg.apply(lambda r: calc_roi(r['revenue'],     r['spend']),          axis=1)
    agg['roi_pct']        = agg['roi'] * 100
    agg['gross_profit']   = agg['revenue'].apply(calc_gross_profit)
    agg['effective_aov']  = agg.apply(lambda r: calc_effective_aov(r['revenue'], r['conversions']), axis=1)
    agg['ltv']            = agg.apply(lambda r: calc_ltv(r['revenue'],     r['conversions']),   axis=1)
    agg['ltv_cac_ratio']  = agg.apply(lambda r: calc_ltv_cac_ratio(r['ltv'], r['cac']),         axis=1)
    agg['payback_months'] = agg.apply(
        lambda r: calc_payback_months(r['cac'], r['revenue'], r['conversions']), axis=1
    )

    # ── Round for readability ─────────────────────────────────────────────
    pct_cols  = ['ctr', 'cvr', 'roi']
    money_cols = ['cpc', 'cpa', 'cac', 'roas', 'effective_aov', 'ltv',
                  'gross_profit', 'spend', 'revenue']
    agg[pct_cols]   = agg[pct_cols].round(4)
    agg[money_cols] = agg[money_cols].round(2)
    agg['roi_pct']  = agg['roi_pct'].round(2)
    agg['ltv_cac_ratio']  = agg['ltv_cac_ratio'].round(2)
    agg['payback_months'] = agg['payback_months'].round(1)
    agg['engagement_score_mean'] = agg['engagement_score_mean'].round(2)

    return agg

# Smoke test on full dataset
full = summarize(df, ['year'])
print(full[['year','n_campaigns','impressions','clicks','conversions',
            'spend','revenue','roi_pct','roas','cpa','ltv_cac_ratio']].to_string(index=False))


 year  n_campaigns  impressions    clicks  conversions        spend      revenue  roi_pct  roas    cpa  ltv_cac_ratio
 2021       200000   1101460304 109954406      8804727 2500878608.0 2.654825e+09     6.16  1.06 284.04           0.64


## 5. Proof: Averaging Row-Level ROI Is Wrong

**The most common analyst mistake:** computing `df['roi_computed'].mean()` instead of  
`(sum_revenue - sum_spend) / sum_spend`.

The two numbers look similar but can differ significantly — and the row-level average is **always biased**  
because it weights every campaign equally regardless of spend size.

A $100 campaign with 500% ROI should not count the same as a $1,000,000 campaign with 6% ROI.  
The ratio-of-sums approach weights campaigns by their spend, which is economically correct.


In [5]:
# ── Global level ─────────────────────────────────────────────────────────
wrong_global = df['roi_computed'].mean()
right_global = calc_roi(df['revenue'].sum(), df['spend'].sum())

print("=" * 60)
print("  PROOF: Row-Level Mean ROI vs Ratio-of-Sums ROI")
print("=" * 60)
print(f"  {'Method':<35} {'ROI':>10}  {'ROI %':>8}")
print("-" * 60)
print(f"  {'WRONG — mean(roi_computed)':<35} {wrong_global:>10.4f}  {wrong_global*100:>7.2f}%")
print(f"  {'RIGHT — (sum_rev - sum_spend)/sum_spend':<35} {right_global:>10.4f}  {right_global*100:>7.2f}%")
print(f"  {'Absolute difference':<35} {abs(wrong_global-right_global):>10.4f}  {abs(wrong_global-right_global)*100:>7.2f}%")
print("=" * 60)
print()
print("The WRONG method overstates portfolio ROI by",
      round((wrong_global - right_global)*100, 2), "percentage points.")
print()

# ── Per-channel level to show the distortion is not uniform ──────────────
print("--- Per-Channel Comparison ---")
wrong_ch = df.groupby('channel')['roi_computed'].mean().round(4)
ch_sums  = df.groupby('channel').agg(rev=('revenue','sum'), spd=('spend','sum'))
right_ch = ((ch_sums['rev'] - ch_sums['spd']) / ch_sums['spd']).round(4)

demo = pd.DataFrame({
    'Wrong_mean_ROI%' : (wrong_ch * 100).round(2),
    'Right_agg_ROI%'  : (right_ch * 100).round(2),
}).assign(Overstatement_pct=lambda x: (x['Wrong_mean_ROI%'] - x['Right_agg_ROI%']).round(2))
print(demo.to_string())
print()
print("Key insight: Wrong method overstates ROI by 9-17 percentage points per channel.")
print("This is because small-spend, high-ROI rows pull the row-level mean upward,")
print("but they represent a tiny share of actual dollars invested.")


  PROOF: Row-Level Mean ROI vs Ratio-of-Sums ROI
  Method                                     ROI     ROI %
------------------------------------------------------------
  WRONG — mean(roi_computed)              0.1991    19.91%
  RIGHT — (sum_rev - sum_spend)/sum_spend     0.0616     6.16%
  Absolute difference                     0.1375    13.75%

The WRONG method overstates portfolio ROI by 13.75 percentage points.

--- Per-Channel Comparison ---


            Wrong_mean_ROI%  Right_agg_ROI%  Overstatement_pct
channel                                                       
Email                 33.59           17.79              15.80
Facebook             -11.57          -22.76              11.19
Google Ads            49.88           34.73              15.15
Instagram             63.89           46.79              17.10
Website               17.76            3.06              14.70
Youtube              -34.76          -43.41               8.65

Key insight: Wrong method overstates ROI by 9-17 percentage points per channel.
This is because small-spend, high-ROI rows pull the row-level mean upward,
but they represent a tiny share of actual dollars invested.


## 6. Three Summary Tables

All three use the same `summarize()` function — consistent methodology across every dimension.


### 6a. Summary by Campaign Type

**Why this cut matters:** Campaign type (Email, Search, Display, Influencer, Social Media) determines creative format and audience intent. Understanding which type drives the best ROI helps allocate the creative budget.


In [6]:
summary_campaign = summarize(df, ['campaign_type'])

display_cols = ['campaign_type', 'n_campaigns', 'spend', 'revenue',
                'roi_pct', 'roas', 'ctr', 'cvr', 'cpa', 'ltv', 'ltv_cac_ratio', 'payback_months']
print("=== Summary by Campaign Type ===")
print(summary_campaign[display_cols].sort_values('roi_pct', ascending=False).to_string(index=False))


=== Summary by Campaign Type ===


campaign_type  n_campaigns       spend      revenue  roi_pct  roas    ctr    cvr    cpa    ltv  ltv_cac_ratio  payback_months
 Social Media        39817 498218100.0 531481943.60     6.68  1.07 0.1002 0.0801 283.33 302.24           0.64            46.9
       Search        40157 501911760.0 533756240.64     6.34  1.06 0.0995 0.0801 284.49 302.54           0.64            47.0
   Influencer        40169 502400525.0 534135601.88     6.32  1.06 0.0998 0.0804 283.64 301.56           0.64            47.0
        Email        39870 498189449.0 527189421.47     5.82  1.06 0.0995 0.0798 285.17 301.77           0.63            47.2
      Display        39987 500158774.0 528261909.53     5.62  1.06 0.1001 0.0801 283.57 299.51           0.63            47.3


### 6b. Summary by Channel

**Why this cut matters:** Channel determines WHERE money is spent. This is the primary table for the business question *"where should we invest more / which should we stop?"*

Sorted by ROI (descending) to surface winners and losers immediately.


In [7]:
summary_channel = summarize(df, ['channel'])

display_cols_ch = ['channel', 'n_campaigns', 'spend', 'revenue', 'roi_pct', 'roas', 'ctr', 'cvr', 'cpa', 'ltv', 'ltv_cac_ratio', 'payback_months']
print('=== Summary by Channel (sorted by ROI) ===')
print(summary_channel[display_cols_ch].sort_values('roi_pct', ascending=False).to_string(index=False))


=== Summary by Channel (sorted by ROI) ===
   channel  n_campaigns       spend      revenue  roi_pct  roas    ctr    cvr    cpa    ltv  ltv_cac_ratio  payback_months
 Instagram        33392 417124850.0 612316844.98    46.79  1.47 0.0997 0.0799 285.13 418.55           0.88            34.1
Google Ads        33438 418912314.0 564394027.79    34.73  1.35 0.0991 0.0801 285.22 384.28           0.81            37.1
     Email        33599 420874104.0 495760405.48    17.79  1.18 0.1001 0.0803 283.34 333.75           0.71            42.4
   Website        33360 416593500.0 429344944.22     3.06  1.03 0.1002 0.0802 281.93 290.56           0.62            48.5
  Facebook        32819 410595258.0 317142246.00   -22.76  0.77 0.0998 0.0802 283.89 219.27           0.46            64.7
   Youtube        33392 416778582.0 235866648.65   -43.41  0.57 0.1000 0.0798 284.75 161.15           0.34            88.4


### 6c. Summary by Year-Month

**Why this cut matters:** Monthly trends reveal seasonality, ramp-up effects, and whether any channel improved or degraded over the year. Critical for budget reallocation decisions.


In [8]:
summary_month = summarize(df, ['year_month'])
summary_month = summary_month.sort_values('year_month')

print("=== Summary by Year-Month ===")
month_display = ['year_month', 'n_campaigns', 'spend', 'revenue', 'roi_pct', 'roas', 'conversions', 'cpa']
print(summary_month[month_display].to_string(index=False))


=== Summary by Year-Month ===
year_month  n_campaigns       spend      revenue  roi_pct  roas  conversions    cpa
   2021-01        16988 212138826.0 225037430.38     6.08  1.06       749513 283.04
   2021-02        15344 191464820.0 204519027.47     6.82  1.07       675518 283.43
   2021-03        16988 211892603.0 225224867.78     6.29  1.06       748406 283.13
   2021-04        16440 205452322.0 217701507.04     5.96  1.06       722244 284.46
   2021-05        16988 211663415.0 224769303.58     6.19  1.06       746884 283.40
   2021-06        16440 205516710.0 217476671.33     5.82  1.06       725194 283.40
   2021-07        16988 212239503.0 227198850.24     7.05  1.07       748763 283.45
   2021-08        16988 213545454.0 223046401.06     4.45  1.04       740767 288.28
   2021-09        16440 206273900.0 219433787.22     6.38  1.06       728936 282.98
   2021-10        16988 212709908.0 225779299.36     6.14  1.06       750733 283.34
   2021-11        16440 205504738.0 219214612.

## 7. LTV, LTV:CAC, and Payback Period Analysis

### Assumptions (documented — must be stated in every presentation)

| Parameter | Value | Source / Rationale |
|-----------|-------|-------------------|
| `new_customer_share` | 60% | Industry benchmark: ~40% of digital conversions are repeat buyers (Klaviyo, 2022) |
| `gross_margin` | 40% | Typical e-commerce gross margin (Shopify, 2022 benchmark) |
| `avg_orders_per_customer` | 2.5 | Conservative repeat-purchase assumption for a mid-market e-commerce brand |
| `avg_months_active` | 12 months | Assumed customer active window for payback calculation |

### What these metrics tell us

- **LTV** = how much revenue one customer generates over their lifetime → higher is better  
- **LTV:CAC** = for every $1 spent acquiring a customer, how many $ come back → benchmark: 3.0x is healthy  
- **Payback months** = how many months to recoup the acquisition cost → shorter is better  

### ⚠️ Important limitation
> These LTV figures are **illustrative**. Real LTV computation requires:
> - Cohort-level retention data (monthly or annual churn rate)
> - Historical repeat-purchase cadence per channel
> - True COGS data to compute real gross margin
>
> **Next step for a real project:** Connect to CRM data (e.g., Salesforce, HubSpot) and compute  
> cohort LTV using the Kaplan-Meier survival method or a BG/NBD model.


In [9]:
print("=== LTV Analysis by Channel ===")
ltv_cols = ['channel', 'n_campaigns', 'conversions', 'effective_aov',
            'cac', 'ltv', 'ltv_cac_ratio', 'payback_months']

ch_ltv = summary_channel[ltv_cols].sort_values('ltv_cac_ratio', ascending=False)
print(ch_ltv.to_string(index=False))
print()

print("=== LTV Benchmarks ===")
print("  LTV:CAC >= 3.0x : Healthy (customer worth 3x acquisition cost)")
print("  LTV:CAC 1-3x    : Marginal — review acquisition spend")
print("  LTV:CAC < 1.0x  : Danger — spending more to acquire than customer ever returns")
print()

for _, row in ch_ltv.iterrows():
    ratio = row['ltv_cac_ratio']
    if ratio >= 3.0:
        verdict = "HEALTHY — scale up"
    elif ratio >= 1.0:
        verdict = "MARGINAL — monitor closely"
    else:
        verdict = "DANGER — losing money on acquisition"
    print(f"  {row['channel']:12s}  LTV:CAC = {ratio:5.2f}x  [{verdict}]")


=== LTV Analysis by Channel ===
   channel  n_campaigns  conversions  effective_aov    cac    ltv  ltv_cac_ratio  payback_months
 Instagram        33392      1462938         418.55 475.21 418.55           0.88            34.1
Google Ads        33438      1468717         384.28 475.37 384.28           0.81            37.1
     Email        33599      1485425         333.75 472.23 333.75           0.71            42.4
   Website        33360      1477660         290.56 469.88 290.56           0.62            48.5
  Facebook        32819      1446338         219.27 473.14 219.27           0.46            64.7
   Youtube        33392      1463649         161.15 474.59 161.15           0.34            88.4

=== LTV Benchmarks ===
  LTV:CAC >= 3.0x : Healthy (customer worth 3x acquisition cost)
  LTV:CAC 1-3x    : Marginal — review acquisition spend
  LTV:CAC < 1.0x  : Danger — spending more to acquire than customer ever returns

  Instagram     LTV:CAC =  0.88x  [DANGER — losing money on ac

## 8. Sanity Checks — Totals Must Match `clean.csv`

**Rule:** The sum of any metric across all rows in each summary table must equal the grand total from `clean.csv`.  
If this fails, there is a groupby error (e.g., rows excluded due to NaN in group key) or a data join issue.

We check five additive metrics across all three summary tables (15 checks total).


In [10]:
ADDITIVE = ['impressions', 'clicks', 'conversions', 'spend', 'revenue']
TOLERANCE = 0.01  # allow $0.01 floating-point rounding tolerance

tables = {
    'summary_campaign' : summary_campaign,
    'summary_channel'  : summary_channel,
    'summary_month'    : summary_month,
}

all_pass = True
print(f"{'Table':<22} {'Metric':<15} {'Expected':>18} {'Got':>18} {'Status'}")
print("-" * 85)

for tname, tbl in tables.items():
    for metric in ADDITIVE:
        expected = GRAND[metric]
        got      = tbl[metric].sum()
        diff     = abs(expected - got)
        status   = "PASS" if diff <= TOLERANCE else f"FAIL (diff={diff:,.4f})"
        if "FAIL" in status:
            all_pass = False
        print(f"{tname:<22} {metric:<15} {expected:>18,.2f} {got:>18,.2f} {status}")

print()
print("n_campaigns checks:")
for tname, tbl in tables.items():
    got = tbl['n_campaigns'].sum()
    exp = GRAND['n_campaigns']
    status = "PASS" if got == exp else f"FAIL (got {got}, expected {exp})"
    print(f"  {tname:<22}: {got:,} / {exp:,} — {status}")
    if "FAIL" in status:
        all_pass = False

print()
print("=" * 50)
print(f"  OVERALL SANITY CHECK: {'ALL PASS' if all_pass else 'ONE OR MORE FAILURES'}")
print("=" * 50)


Table                  Metric                    Expected                Got Status
-------------------------------------------------------------------------------------
summary_campaign       impressions       1,101,460,304.00   1,101,460,304.00 PASS
summary_campaign       clicks              109,954,406.00     109,954,406.00 PASS
summary_campaign       conversions           8,804,727.00       8,804,727.00 PASS
summary_campaign       spend             2,500,878,608.00   2,500,878,608.00 PASS
summary_campaign       revenue           2,654,825,117.12   2,654,825,117.12 PASS
summary_channel        impressions       1,101,460,304.00   1,101,460,304.00 PASS
summary_channel        clicks              109,954,406.00     109,954,406.00 PASS
summary_channel        conversions           8,804,727.00       8,804,727.00 PASS
summary_channel        spend             2,500,878,608.00   2,500,878,608.00 PASS
summary_channel        revenue           2,654,825,117.12   2,654,825,117.12 PASS
summary_mo

## 9. Save Summary Tables


In [11]:
paths = {
    'summary_campaign' : PRO_DIR / 'summary_campaign.csv',
    'summary_channel'  : PRO_DIR / 'summary_channel.csv',
    'summary_month'    : PRO_DIR / 'summary_month.csv',
}

summary_campaign.to_csv(paths['summary_campaign'], index=False)
summary_channel.to_csv(paths['summary_channel'],   index=False)
summary_month.to_csv(paths['summary_month'],        index=False)

for name, path in paths.items():
    tbl = pd.read_csv(path)
    print(f"Saved {path.name:30s}  shape={tbl.shape}")


Saved summary_campaign.csv            shape=(5, 21)
Saved summary_channel.csv             shape=(6, 21)
Saved summary_month.csv               shape=(12, 21)


## 10. Channel Summary — Plain English

This is the business-ready translation of `summary_channel.csv`.  
Use this language when presenting to a non-technical stakeholder.


In [12]:
print("=" * 70)
print("  CHANNEL PERFORMANCE SUMMARY — Plain English")
print("  (Simulated Data — Portfolio Project)")
print("=" * 70)
print()

ch = summary_channel.sort_values('roi_pct', ascending=False).reset_index(drop=True)
for _, row in ch.iterrows():
    roi_sign = "+" if row['roi_pct'] >= 0 else ""
    print(f"  {row['channel']:12s}")
    print(f"    Spend   : ${row['spend']:>14,.0f}")
    print(f"    Revenue : ${row['revenue']:>14,.0f}")
    print(f"    ROI     : {roi_sign}{row['roi_pct']:.1f}%  |  ROAS: {row['roas']:.2f}x")
    print(f"    CPA     : ${row['cpa']:>10,.2f}  |  LTV:CAC: {row['ltv_cac_ratio']:.2f}x")
    print(f"    Payback : {row['payback_months']:.1f} months")
    print()


  CHANNEL PERFORMANCE SUMMARY — Plain English
  (Simulated Data — Portfolio Project)

  Instagram   
    Spend   : $   417,124,850
    Revenue : $   612,316,845
    ROI     : +46.8%  |  ROAS: 1.47x
    CPA     : $    285.13  |  LTV:CAC: 0.88x
    Payback : 34.1 months

  Google Ads  
    Spend   : $   418,912,314
    Revenue : $   564,394,028
    ROI     : +34.7%  |  ROAS: 1.35x
    CPA     : $    285.22  |  LTV:CAC: 0.81x
    Payback : 37.1 months

  Email       
    Spend   : $   420,874,104
    Revenue : $   495,760,405
    ROI     : +17.8%  |  ROAS: 1.18x
    CPA     : $    283.34  |  LTV:CAC: 0.71x
    Payback : 42.4 months

  Website     
    Spend   : $   416,593,500
    Revenue : $   429,344,944
    ROI     : +3.1%  |  ROAS: 1.03x
    CPA     : $    281.93  |  LTV:CAC: 0.62x
    Payback : 48.5 months

  Facebook    
    Spend   : $   410,595,258
    Revenue : $   317,142,246
    ROI     : -22.8%  |  ROAS: 0.77x
    CPA     : $    283.89  |  LTV:CAC: 0.46x
    Payback : 64.7 mon

## 11. Findings — Phase 3 Summary

### Key Metrics (Portfolio-Level)

| Metric | Value |
|--------|-------|
| Total Spend | $2.50B |
| Total Revenue | $2.65B |
| Net Profit | $154M |
| Portfolio ROI | **+6.2%** |
| Portfolio ROAS | **1.06x** |
| Mean CPA | $284 |

### Channel Verdict

| Channel | ROI | ROAS | Verdict |
|---------|-----|------|---------|
| Instagram | +46.8% | 1.47x | **Scale up** — best return per dollar |
| Google Ads | +34.7% | 1.35x | **Invest more** — reliable, high-volume |
| Email | +17.8% | 1.18x | **Maintain** — solid, low-risk |
| Website | +3.1% | 1.03x | **Monitor** — barely profitable |
| Facebook | -22.8% | 0.77x | **Reduce budget** — losing money |
| YouTube | -43.4% | 0.57x | **Stop or repurpose** — worst ROI |

### LTV:CAC Interpretation
- Instagram and Google Ads show LTV:CAC > 3x — healthy, should receive more budget
- Facebook and YouTube show LTV:CAC < 1x — spending more to acquire customers than they return

### ⚠️ Caveats to state with every table
1. Revenue is simulated. Relative channel ranking is the insight, not absolute dollar values.
2. LTV figures use assumed margin (40%) and repeat-purchase rate (2.5x) — flag these in presentations.
3. Attribution is last-click. YouTube's top-of-funnel assist value is invisible here.
4. CAC assumes 60% of conversions are new customers. Real businesses track this separately.
